In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%load_ext autoreload
%autoreload 2

from src.data import load_train
from src.config import FAMILIES, ATTACK_TO_FAMILY, METRICS_DIR, FIGURES_DIR
from src.evaluation import load_oof, wilson_ci, per_attack_recall, missed_destinations, fp_breakdown

X_train, y_train, meta_train = load_train()

# ---- ALIGNMENT TEST (the classic bug) ----
# OOF arrays are positional: oof[i] <-> X_train.iloc[i] <-> y_train.iloc[i] <-> meta_train.iloc[i].
derived = meta_train.map(ATTACK_TO_FAMILY)
assert (derived.values == y_train.values).all(), \
    "ALIGNMENT BUG: ATTACK_TO_FAMILY[meta_train.label] != y_train"
print(f"alignment OK for all {len(y_train)} rows: meta_train -> family == y_train")

# Positional arrays aligned with OOF outputs
y_true = y_train.values          # family
meta_true = meta_train.values    # attack name
print("train family counts:", dict(pd.Series(y_true).value_counts()))

In [ ]:
from sklearn.metrics import confusion_matrix

MODELS = ["M1_logreg", "M2_tree", "M3_rf", "M4_lgbm"]
oof = {m: load_oof(m) for m in MODELS}   # {model: (y_pred, y_proba)}

fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4))
for ax, m in zip(axes, MODELS):
    yp, _ = oof[m]
    cm = confusion_matrix(y_true, yp, labels=FAMILIES)
    # structural checks
    assert cm.sum() == len(X_train), f"{m}: confusion total != {len(X_train)}"
    expected_rows = pd.Series(y_true).value_counts().reindex(FAMILIES).values
    assert np.array_equal(cm.sum(axis=1), expected_rows), f"{m}: row sums wrong"
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, ax=ax, annot=True, fmt=".2f", xticklabels=FAMILIES,
                yticklabels=FAMILIES, cmap="Blues", cbar=False, vmin=0, vmax=1)
    ax.set_title(m)
plt.suptitle("Pooled OOF confusion (row-normalized)")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "p15_confusion_pooled.png", dpi=120); plt.show()
print("confusion totals and row sums verified for all models")

In [ ]:
R2L_ATTACKS = [a for a, f in ATTACK_TO_FAMILY.items() if f == "R2L"]
U2R_ATTACKS = [a for a, f in ATTACK_TO_FAMILY.items() if f == "U2R"]
print("R2L attacks:", R2L_ATTACKS)
print("U2R attacks:", U2R_ATTACKS)

for family in ["R2L", "U2R"]:
    for m in MODELS:
        yp, _ = oof[m]
        tbl = per_attack_recall(meta_true, yp, family, ATTACK_TO_FAMILY)
        tbl.to_csv(METRICS_DIR / f"p15_per_attack_{m}_{family}.csv", index=False)
        print(f"\n=== {m} — {family} (pooled OOF) ===")
        print(tbl.to_string(index=False))
        # per-attack n must sum to the family count
        fam_n = int((y_true == family).sum())
        assert tbl["n"].sum() == fam_n, f"{m}/{family}: attack n doesn't sum to {fam_n}"
print("\nper-attack n sums to family counts for every model")

In [ ]:
for family in ["U2R", "R2L"]:
    print(f"\n=== Missed {family} destinations ===")
    for m in ["M2_tree", "M3_rf", "M4_lgbm"]:
        yp, _ = oof[m]
        dest = missed_destinations(y_true, yp, family)
        total_missed = int(dest.sum())
        to_normal = int(dest.get("Normal", 0))
        print(f"  {m:<10} missed={total_missed:>3}  -> Normal={to_normal:>3} "
              f"({to_normal/total_missed:.0%})  | {dest.to_dict()}")

In [ ]:
for m in ["M4_lgbm", "M3_rf"]:
    yp, _ = oof[m]
    fp_df, n_fp = fp_breakdown(yp, y_true, X_train, by=("service", "flag"))
    print(f"\n=== {m}: {n_fp} false positives ===")
    for col in ["service", "flag"]:
        sub = fp_df[fp_df["by"] == col].head(8)
        print(f" by {col}:")
        print(sub[[col, "n_fp", "share_of_fp"]].to_string(index=False))

In [ ]:
for m in ["M3_rf", "M4_lgbm"]:
    fd = pd.read_csv(METRICS_DIR / f"cv_results_{m}.csv")
    print(f"{m} per-fold U2R recall: {fd['recall_U2R'].round(3).tolist()}")
    print(f"  -> each fold validates on ~{(y_true=='U2R').sum()//5} U2R rows, "
          f"so one miss swings recall by ~{1/((y_true=='U2R').sum()//5):.0%}")

In [ ]:
m = "M4_lgbm"
yp, _ = oof[m]
u2r = y_true == "U2R"
caught = u2r & (yp == "U2R")
missed = u2r & (yp != "U2R")
normal = y_true == "Normal"

probe_cols = ["duration", "src_bytes", "dst_bytes", "hot", "num_root",
              "num_file_creations", "srv_count"]
print(f"U2R caught={int(caught.sum())}, missed={int(missed.sum())}")
rows = []
for c in probe_cols:
    rows.append({"feature": c,
                 "missed_U2R_mean": X_train.iloc[missed][c].mean(),
                 "caught_U2R_mean": X_train.iloc[caught][c].mean(),
                 "Normal_mean": X_train.iloc[normal][c].mean()})
print(pd.DataFrame(rows).round(2).to_string(index=False))

In [ ]:
from sklearn.base import clone
from src.models import get_models

pipe_m2 = clone(get_models()["M2_tree"])
pipe_m2.fit(X_train, y_train)
tree = pipe_m2.named_steps["clf"]
X_t = pipe_m2.named_steps["pre"].transform(X_train)
leaves = tree.apply(X_t)

u2r_idx = np.where(y_true == "U2R")[0]
u2r_leaves = leaves[u2r_idx]
uniq, counts = np.unique(leaves, return_counts=True)
leaf_size = dict(zip(uniq, counts))

print("Leaves used by U2R rows (tests Entry 9c 'memorizes tiny U2R leaves'):")
for lf in np.unique(u2r_leaves):
    n_u2r = int((u2r_leaves == lf).sum())
    print(f"  leaf {lf}: total size={leaf_size[lf]:>3}, U2R rows={n_u2r}")